# **Fase 5b · Dossiês de escuta**
### *Envia as entradas ao Gemini, registra cada chamada, monta os dossiês e calcula as medidas, conforme as seções 2 e 4 a 7 de `docs/fase5/01_desenho_experimental.md`.*

(Este caderno não usa os áudios nem a GPU. As seções 4 e 5 chamam o Gemini com a chave guardada nos Secrets do Colab, com o nome `GEMINI_API_KEY`, que nunca aparece no caderno. As seções 6 a 8 leem apenas os registros gravados das chamadas e podem ser executadas por qualquer pessoa a partir do repositório, sem chave)

*Execução.* As seções 1 a 7, 9 e 10 foram executadas em 02 e 03/11/2026, e as chamadas ao Gemini, em 03/11. Depois da leitura dos dossiês pela pesquisadora, as seções 1 a 3, 6 e 8 foram executadas de novo, em 13/11/2026, num ambiente novo e sem nenhuma chamada nova (a seção 1 registra as 16 chamadas já gravadas). Por isso a numeração das células executadas não é contínua.

## **1. Preparação**

*Confere a biblioteca do Google para a API do Gemini (`google-genai`), já instalada no Colab, e se ela aceita o esquema da saída. A biblioteca não é reinstalada numa versão fixada, porque a versão mais recente exige uma versão do `google-auth` diferente da que o próprio Colab usa, e o conflito poderia afetar a conexão com o Drive e com os Secrets. A versão usada fica gravada em cada chamada e na seção 9. Depois, conecta o Drive, define as pastas e lê os cinco arquivos de entrada gravados pelo caderno 5a. As funções de leitura da tabela de referência são as da fase 4, em `src/eventos.py`.*

In [1]:
import importlib.metadata as md
from google.genai import types
print('google-genai', md.version('google-genai'), '· google-auth', md.version('google-auth'))
print('Aceita o esquema da saída:', 'response_json_schema' in types.GenerateContentConfig.model_fields)

google-genai 2.12.1 · google-auth 2.49.0
Aceita o esquema da saída: True


**Resultado.** A versão instalada no Colab (2.12.1) aceita o esquema da saída. A tentativa de fixar a versão 2.34.0, em 02/11, gerou conflito com o `google-auth` usado pelo próprio Colab (diário, seção 1).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, sys, json, time
from datetime import datetime, timezone
from itertools import combinations
import pandas as pd

PASTA_PROJETO = '/content/drive/MyDrive/escuta-cerrada-poc'
PASTA_FASE5 = os.path.join(PASTA_PROJETO, 'resultados', 'fase5')
PASTA_ENTRADAS = os.path.join(PASTA_FASE5, 'entradas')
PASTA_CHAMADAS = os.path.join(PASTA_FASE5, 'chamadas')
PASTA_DOSSIES = os.path.join(PASTA_FASE5, 'dossies')
PASTA_MEDIDAS = os.path.join(PASTA_FASE5, 'medidas')
PASTA_REFERENCIAS = os.path.join(PASTA_PROJETO, 'referencias')
for pasta in [PASTA_CHAMADAS, PASTA_DOSSIES, PASTA_MEDIDAS]:
    os.makedirs(pasta, exist_ok=True)

sys.path.insert(0, os.path.join(PASTA_PROJETO, 'src'))
from eventos import ler_referencia


def ler(caminho):
    with open(caminho, encoding='utf-8') as f:
        return json.load(f)


ENTRADAS = {n: ler(os.path.join(PASTA_ENTRADAS, f'faixa{n:02d}.json')) for n in range(1, 6)}
print(len(ENTRADAS), 'entradas lidas ·', len(os.listdir(PASTA_CHAMADAS)), 'chamadas já registradas')

Mounted at /content/drive
5 entradas lidas · 16 chamadas já registradas


### 1.1 Chave de acesso

Lê a chave nos Secrets do Colab e cria a conexão com a API. O valor da chave não é exibido nem gravado. Na primeira execução, o Colab pergunta se este caderno pode usar o secret `GEMINI_API_KEY`, e é preciso conceder o acesso. Sem a chave, a célula avisa e as seções 6 a 8 continuam executáveis a partir dos registros.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

try:
    CLIENTE = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    print('Chave encontrada nos Secrets. O valor não é exibido.')
except Exception as erro:
    CLIENTE = None
    print('Chave não disponível (' + type(erro).__name__ + '). As seções 6 a 8 podem ser executadas a partir dos registros.')

Chave encontrada nos Secrets. O valor não é exibido.


**Resultado.** A chave foi lida dos Secrets sem ser exibida.

## **2. Modelo, instrução, regras e esquema**

*Fixa o modelo e transcreve para o código o anexo A (instrução de sistema), a lista de motivos e a regra do grau de apoio (seções 4 e 5) e o esquema da saída. A temperatura e os demais parâmetros ficam no valor padrão do modelo, como prevê a seção 2. No esquema, os motivos e os graus são listas fechadas, de modo que a própria API só aceita respostas com esses valores. Um teste automático (seção 10) confere que a instrução do código é idêntica, palavra por palavra, ao anexo A do desenho.*

In [3]:
MODELO = 'gemini-3.8-flash'

INSTRUCAO = (
    'Você recebe as saídas de modelos automáticos aplicados a uma gravação sonora em disco. Você não '
    'tem acesso ao áudio. Sua tarefa é indicar a uma pesquisadora os momentos da gravação em que vale '
    'escutar com atenção, com base apenas nos itens da entrada.\n\n'
    '1. Cada indicação deve citar os identificadores dos itens da entrada que a sustentam. O início e o '
    'fim da indicação devem ficar dentro do intervalo desses itens.\n'
    '2. Descreva o que os modelos registraram, e não o que soa. Por exemplo, escreva “o modelo de fala '
    'escreveu o mesmo texto em seis segmentos seguidos” ou “o detector de eventos atribuiu 0,82 ao '
    'rótulo Applause”, e não “há aplausos” ou “o orador repete a frase”.\n'
    '3. Não interprete sentido, intenção, ironia, emoção, gênero, tema ou contexto da gravação, e não '
    'deduza o que ela representa.\n'
    '4. Use apenas os motivos da lista fornecida e atribua o grau de apoio segundo a regra fornecida.\n'
    '5. Se um trecho não tiver nada a indicar, não crie indicação para ele.\n'
    '6. A frase de cada indicação deve dizer o que a pesquisadora pode verificar na escuta, em uma ou '
    'duas frases, em português.'
)


MOTIVOS = [
    'repetição na transcrição',
    'texto com indicação de ausência de fala',
    'transcrição com baixa confiança',
    'divergência entre transcrições',
    'evento sonoro apontado',
    'coincidência entre fontes',
    'rótulo isolado',
]


GRAUS = ['alto', 'médio', 'baixo']


REGRAS_PEDIDO = (
    'Motivos possíveis: ' + '; '.join(MOTIVOS[:-1]) + '; rótulo isolado (rótulo ou descrição que '
    'aparece uma única vez na faixa).\n\n'
    'Regra do grau de apoio:\n'
    '- alto: dois ou mais itens de fontes diferentes (fala, PANNs, CLAP) coincidem no intervalo, ou um '
    'rótulo do PANNs tem probabilidade de pelo menos 0,5;\n'
    '- médio: um único item do PANNs com probabilidade entre 0,3 e 0,5, ou um único segmento de fala '
    'com algum dos quatro sinais;\n'
    '- baixo: apenas uma descrição do CLAP.'
)


ESQUEMA = {
    'type': 'object',
    'properties': {
        'indicacoes': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'inicio': {'type': 'number', 'description': 'início da indicação, em segundos'},
                    'fim': {'type': 'number', 'description': 'fim da indicação, em segundos'},
                    'fontes': {'type': 'array', 'items': {'type': 'string'},
                               'description': 'identificadores dos itens da entrada que sustentam a indicação'},
                    'motivo': {'type': 'string', 'enum': MOTIVOS},
                    'grau_de_apoio': {'type': 'string', 'enum': GRAUS},
                    'frase': {'type': 'string', 'description': 'o que a pesquisadora pode verificar na escuta'},
                },
                'required': ['inicio', 'fim', 'fontes', 'motivo', 'grau_de_apoio', 'frase'],
            },
        },
    },
    'required': ['indicacoes'],
}


def monta_pedido(entrada):
    """Texto enviado ao modelo, além da instrução de sistema: regras e entrada da faixa."""
    return (REGRAS_PEDIDO + f'\n\nEntrada da faixa {entrada["faixa"]}:\n'
            + json.dumps(entrada, ensure_ascii=False, indent=1))


print(INSTRUCAO)
print('\n' + '·' * 60 + '\n')
print(REGRAS_PEDIDO)

Você recebe as saídas de modelos automáticos aplicados a uma gravação sonora em disco. Você não tem acesso ao áudio. Sua tarefa é indicar a uma pesquisadora os momentos da gravação em que vale escutar com atenção, com base apenas nos itens da entrada.

1. Cada indicação deve citar os identificadores dos itens da entrada que a sustentam. O início e o fim da indicação devem ficar dentro do intervalo desses itens.
2. Descreva o que os modelos registraram, e não o que soa. Por exemplo, escreva “o modelo de fala escreveu o mesmo texto em seis segmentos seguidos” ou “o detector de eventos atribuiu 0,82 ao rótulo Applause”, e não “há aplausos” ou “o orador repete a frase”.
3. Não interprete sentido, intenção, ironia, emoção, gênero, tema ou contexto da gravação, e não deduza o que ela representa.
4. Use apenas os motivos da lista fornecida e atribua o grau de apoio segundo a regra fornecida.
5. Se um trecho não tiver nada a indicar, não crie indicação para ele.
6. A frase de cada indicação de

**Resultado.** A instrução é idêntica ao anexo A do desenho, o que o teste `test_instrucao_e_o_texto_do_anexo_a` confere na seção 10.

### 2.1 O pedido de uma faixa

Mostra o começo do texto que acompanha a instrução de sistema na chamada da faixa 04, a menor. Ele traz as regras e, em seguida, o arquivo de entrada tal como foi gravado pelo caderno 5a.

In [ ]:
pedido = monta_pedido(ENTRADAS[4])
print(len(pedido), 'caracteres\n')
print(pedido[:2500], '\n[...]')

5731 caracteres

Motivos possíveis: repetição na transcrição; texto com indicação de ausência de fala; transcrição com baixa confiança; divergência entre transcrições; evento sonoro apontado; coincidência entre fontes; rótulo isolado (rótulo ou descrição que aparece uma única vez na faixa).

Regra do grau de apoio:
- alto: dois ou mais itens de fontes diferentes (fala, PANNs, CLAP) coincidem no intervalo, ou um rótulo do PANNs tem probabilidade de pelo menos 0,5;
- médio: um único item do PANNs com probabilidade entre 0,3 e 0,5, ou um único segmento de fala com algum dos quatro sinais;
- baixo: apenas uma descrição do CLAP.

Entrada da faixa 4:
{
 "faixa": 4,
 "duracao_s": 64.86,
 "legenda": {
  "fala": "Segmentos produzidos pelo modelo de reconhecimento de fala Whisper (large-v3), sem filtro de ruído, com início e fim em segundos, o texto escrito pelo modelo e três indicadores do próprio modelo. avg_logprob é a média do logaritmo da probabilidade dos tokens do texto (valores mais baix

## **3. Chamada e registro**

*Cada chamada é gravada num arquivo próprio, em `resultados/fase5/chamadas`, com data, modelo e versão informada pela API, parâmetros, instrução, esquema, pedido, resposta, tempo de resposta e tokens de entrada, de saída e de raciocínio. Se o arquivo de uma chamada já existe, ela não é refeita, o que permite executar de novo a célula sem gerar chamadas a mais. Quando o servidor recusa a chamada por sobrecarga (erros 503 e 504), a função espera 2 minutos e tenta de novo, até três vezes, e registra as falhas. Quando a recusa vem da cota da conta (erro 429), a função para na hora, porque cada nova tentativa também consome a cota. Cada tentativa tem um tempo limite de 5 minutos, depois do qual é considerada recusada, porque a biblioteca, por padrão, espera a resposta sem limite.*

In [4]:
PAUSA = 15  # segundos entre chamadas, por precaução com o limite da camada gratuita


def sem_chave(texto):
    """Apaga da mensagem de erro qualquer trecho com aparência de chave."""
    return re.sub(r'(key=)[^&\s]+', r'\1***', texto)


def chamar(pedido, nome):
    caminho = os.path.join(PASTA_CHAMADAS, nome + '.json')
    if os.path.exists(caminho):
        print(nome, '· já registrada, não foi chamada de novo')
        return ler(caminho)
    if CLIENTE is None:
        raise RuntimeError('Sem chave de acesso (seção 1.1).')
    config = types.GenerateContentConfig(system_instruction=INSTRUCAO, response_mime_type='application/json',
                                         response_json_schema=ESQUEMA,
                                         http_options=types.HttpOptions(timeout=300_000))
    falhas = []
    for tentativa in range(1, 4):
        inicio = time.time()
        try:
            resposta = CLIENTE.models.generate_content(model=MODELO, contents=pedido, config=config)
            break
        except Exception as erro:
            falhas.append({'tentativa': tentativa, 'erro': sem_chave(f'{type(erro).__name__}: {erro}')[:600]})
            print(nome, '· tentativa', tentativa, 'recusada:', falhas[-1]['erro'][:200])
            if getattr(erro, 'code', None) == 429:
                raise RuntimeError(nome + ' · cota da API esgotada (erro 429). Nada foi gravado. '
                                   'Novas tentativas agora só consumiriam a cota, por isso o caderno para aqui.')
            if tentativa < 3:
                time.sleep(120)
    else:
        raise RuntimeError(nome + ' · três tentativas sem resposta. Nada foi gravado.')
    tempo = time.time() - inicio
    uso = resposta.usage_metadata
    try:
        saida, erro_leitura = json.loads(resposta.text), None
    except Exception as erro:
        saida, erro_leitura = None, f'{type(erro).__name__}: {erro}'
    registro = {
        'chamada': nome,
        'data_execucao': datetime.now(timezone.utc).isoformat(),
        'modelo': MODELO,
        'versao_modelo': resposta.model_version,
        'biblioteca': 'google-genai ' + genai.__version__,
        'parametros': {'temperatura': 'padrão do modelo', 'saida': 'application/json, com o esquema registrado',
                       'demais': 'valores padrão',
                       'tempo_limite_s': 300},
        'instrucao': INSTRUCAO,
        'esquema': ESQUEMA,
        'pedido': pedido,
        'resposta_texto': resposta.text,
        'resposta': saida,
        'erro_leitura': erro_leitura,
        'termino': str(resposta.candidates[0].finish_reason) if resposta.candidates else None,
        'tentativas_recusadas': falhas,
        'tempo_resposta_s': round(tempo, 2),
        'tokens': {'entrada': uso.prompt_token_count, 'saida': uso.candidates_token_count,
                   'raciocinio': uso.thoughts_token_count, 'total': uso.total_token_count},
    }
    with open(caminho, 'w', encoding='utf-8') as f:
        json.dump(registro, f, ensure_ascii=False, indent=1)
    return registro


def resumo(registro):
    saida = registro['resposta'] or {}
    return {'chamada': registro['chamada'], 'indicações': len(saida.get('indicacoes', [])),
            'erro de leitura': registro['erro_leitura'] or '', 'tempo (s)': registro['tempo_resposta_s'],
            **{f'tokens {k}': v for k, v in registro['tokens'].items()}}

## **4. Teste de funcionamento**

*Uma única chamada com uma entrada fictícia, de dados inventados e fora do corpus, para conferir a conexão, o esquema e o registro antes das chamadas com as faixas. A entrada imita o formato do caderno 5a, com a mesma legenda, um texto repetido em três segmentos, um rótulo do PANNs e uma descrição do CLAP. O registro fica em `chamadas/teste.json` e não entra nas medidas.*

In [ ]:
ENTRADA_TESTE = {
    'faixa': 0,
    'duracao_s': 30.0,
    'legenda': ENTRADAS[1]['legenda'],
    'fala': [
        {'id': 'F-001', 'inicio': 0.0, 'fim': 4.0, 'texto': 'Bom dia a todos os presentes.', 'avg_logprob': -0.2,
         'no_speech_prob': 0.05, 'compression_ratio': 1.1, 'sinais': []},
        {'id': 'F-002', 'inicio': 5.0, 'fim': 8.0, 'texto': 'O TREM DAS SETE', 'avg_logprob': -0.3,
         'no_speech_prob': 0.7, 'compression_ratio': 0.7, 'sinais': ['repetição', 'indicação de ausência de fala']},
        {'id': 'F-003', 'inicio': 8.0, 'fim': 11.0, 'texto': 'O TREM DAS SETE', 'avg_logprob': -0.3,
         'no_speech_prob': 0.1, 'compression_ratio': 0.7, 'sinais': ['repetição']},
        {'id': 'F-004', 'inicio': 11.0, 'fim': 14.0, 'texto': 'O TREM DAS SETE', 'avg_logprob': -1.4,
         'no_speech_prob': 0.1, 'compression_ratio': 0.7, 'sinais': ['repetição', 'baixa confiança']},
    ],
    'eventos_panns': [{'id': 'P-001', 'inicio': 18.0, 'fim': 21.0, 'rotulo': 'Dog', 'probabilidade_maxima': 0.72}],
    'resumo_panns': 'Rótulos apontados em mais da metade das janelas da faixa, que não entram como itens: Speech (70% das janelas).',
    'descricoes_clap': [{'id': 'C-001', 'inicio': 24.0, 'fim': 26.0, 'descricao': 'a church bell'}],
}

TESTE = chamar(monta_pedido(ENTRADA_TESTE), 'teste')
print(pd.DataFrame([resumo(TESTE)]).to_string(index=False))
print('\nVersão do modelo informada pela API:', TESTE['versao_modelo'])
print(json.dumps(TESTE['resposta'], ensure_ascii=False, indent=1))

chamada  indicações erro de leitura  tempo (s)  tokens entrada  tokens saida  tokens raciocinio  tokens total
  teste           5                      43.04            1574           649               4710          6933

Versão do modelo informada pela API: gemini-3.8-flash
{
 "indicacoes": [
  {
   "inicio": 5.0,
   "fim": 8.0,
   "fontes": [
    "F-002"
   ],
   "motivo": "texto com indicação de ausência de fala",
   "grau_de_apoio": "médio",
   "frase": "O modelo de fala transcreveu o texto com probabilidade de ausência de fala de 0,7. A pesquisadora pode verificar na escuta se há presença efetiva de voz ou apenas silêncio e ruído no trecho."
  },
  {
   "inicio": 8.0,
   "fim": 11.0,
   "fontes": [
    "F-003"
   ],
   "motivo": "repetição na transcrição",
   "grau_de_apoio": "médio",
   "frase": "O modelo de fala repetiu o mesmo texto que ocorre em outros segmentos da gravação. A pesquisadora pode verificar na escuta se o conteúdo sonoro corresponde a uma repetição real."
  },
  {

**Resultado.** O teste foi recusado pelo servidor em 02/11 (erros 503 e 504, por sobrecarga) e, em 03/11, pela cota da camada gratuita (erro 429), e só obteve resposta em 03/11, já na camada paga (diário, seção 1). A resposta veio no formato do esquema, com 5 indicações, todas com fontes existentes, tempos iguais aos das fontes e graus segundo a regra, e o segmento sem sinal não foi indicado. Os três segmentos com o mesmo texto receberam três indicações separadas, cada uma com um motivo, e as frases relatam o registro para, em seguida, traduzir o rótulo num som a verificar (“latidos”, “badaladas de sino”). A chamada levou 43 s e usou 4.710 tokens de raciocínio para 649 de saída.

## **5. Chamadas com as faixas**

*As cinco faixas, três vezes cada, com a mesma entrada e os mesmos parâmetros, num total de quinze chamadas (seção 6 do desenho). Entre uma chamada e outra há uma pausa de 15 segundos. A célula leva alguns minutos. Se for interrompida, basta executá-la de novo, porque as chamadas já registradas não são refeitas.*

In [ ]:
for n in range(1, 6):
    for rodada in range(1, 4):
        nome = f'faixa{n:02d}_{rodada}'
        nova = not os.path.exists(os.path.join(PASTA_CHAMADAS, nome + '.json'))
        registro = chamar(monta_pedido(ENTRADAS[n]), nome)
        if nova:
            print(nome, '·', resumo(registro)['indicações'], 'indicações ·', registro['tempo_resposta_s'], 's')
            time.sleep(PAUSA)

faixa01_1 · 12 indicações · 51.32 s
faixa01_2 · 13 indicações · 51.26 s
faixa01_3 · 10 indicações · 61.21 s
faixa02_1 · 6 indicações · 39.69 s
faixa02_2 · 7 indicações · 60.95 s
faixa02_3 · 7 indicações · 38.45 s
faixa03_1 · 4 indicações · 115.57 s
faixa03_2 · 4 indicações · 50.81 s
faixa03_3 · 4 indicações · 63.53 s
faixa04_1 · 3 indicações · 263.61 s
faixa04_2 · 4 indicações · 47.46 s
faixa04_3 · 4 indicações · 24.79 s
faixa05_1 · 7 indicações · 31.62 s
faixa05_2 · 5 indicações · 30.13 s
faixa05_3 · 8 indicações · 48.83 s


**Resultado.** As quinze chamadas foram feitas em 03/11/2026, sem nenhuma recusa. O número de indicações por chamada variou de 3 a 13, e o tempo de resposta, de 25 a 264 s.

## **6. Indicações e conferência automática**

*A partir daqui o caderno lê apenas os registros gravados. As funções transcrevem as medidas 1 a 5 da seção 7. A conformidade verifica tempos, fontes, motivo, grau e frase. A invenção verifica identificadores inexistentes e intervalos fora do intervalo das fontes citadas, com tolerância de 1 segundo, entendido como o trecho entre o menor início e o maior fim dessas fontes. A restrição procura nas frases palavras iniciadas pelos termos da lista (“tristeza” conta como “triste”), e a leitura da pesquisadora decide depois o que de fato interpreta. O grau esperado aplica a regra da seção 4 às fontes citadas. Quando as fontes formam uma combinação que a regra não prevê (por exemplo, um segmento de fala sem sinal, ou fontes diferentes que não coincidem no tempo), o grau esperado fica em branco, e a indicação sai do cálculo da coerência. Duas indicações de chamadas diferentes correspondem quando têm o mesmo motivo e algum tempo em comum.*

In [5]:
TERMOS_INTERPRETATIVOS = [
    'ironia', 'irônico', 'emoção', 'emocionado', 'sentimento', 'triste', 'alegre', 'entusiasmo',
    'intenção', 'pretende', 'quer dizer', 'significa', 'simboliza', 'representa', 'sugere que', 'revela',
]


TOLERANCIA = 1.0  # segundos, medida 2 (invenção)


def mapa_fontes(entrada):
    """{identificador: item} com os itens das três fontes."""
    return {i['id']: i for bloco in ['fala', 'eventos_panns', 'descricoes_clap'] for i in entrada[bloco]}


def sobrepoe(a, b):
    """Os intervalos a e b têm algum tempo em comum (contato num ponto conta)."""
    return a['inicio'] <= b['fim'] and b['inicio'] <= a['fim']


def problemas_conformidade(ind):
    """Lista do que falta ou é inválido numa indicação (lista vazia = conforme)."""
    p = []
    num = all(isinstance(ind.get(k), (int, float)) and not isinstance(ind.get(k), bool) for k in ['inicio', 'fim'])
    if not num:
        p.append('tempo ausente ou não numérico')
    elif not 0 <= ind['inicio'] <= ind['fim']:
        p.append('início depois do fim')
    f = ind.get('fontes')
    if not isinstance(f, list) or not f or not all(isinstance(x, str) and x for x in f):
        p.append('sem fonte')
    if ind.get('motivo') not in MOTIVOS:
        p.append('motivo fora da lista')
    if ind.get('grau_de_apoio') not in GRAUS:
        p.append('grau fora da lista')
    if not isinstance(ind.get('frase'), str) or not ind['frase'].strip():
        p.append('sem frase')
    return p


def invencao(ind, fontes):
    """(identificadores inexistentes, intervalo fora das fontes citadas com tolerância)."""
    citados = ind.get('fontes') or []
    inexistentes = [i for i in citados if i not in fontes]
    validos = [fontes[i] for i in citados if i in fontes]
    numericos = all(isinstance(ind.get(k), (int, float)) and not isinstance(ind.get(k), bool) for k in ['inicio', 'fim'])
    fora = False
    if validos and numericos:
        fora = (ind['inicio'] < min(v['inicio'] for v in validos) - TOLERANCIA
                or ind['fim'] > max(v['fim'] for v in validos) + TOLERANCIA)
    return inexistentes, fora


def termos_na_frase(frase):
    """Palavras da frase que começam por algum termo da lista (por exemplo, “tristeza” por “triste”)."""
    achados = []
    texto = (frase or '').lower()
    for termo in TERMOS_INTERPRETATIVOS:
        for m in re.finditer(r'(?<!\w)' + re.escape(termo) + r'\w*', texto):
            achados.append(m.group(0))
    return achados


def grau_esperado(ind, fontes):
    """Grau de apoio que a regra da seção 4 atribui às fontes citadas, ou None quando a regra não
    se aplica (identificador inexistente, ou combinação de fontes que a regra não prevê)."""
    citados = ind.get('fontes') or []
    if not citados or any(i not in fontes for i in citados):
        return None
    itens = [(i[0], fontes[i]) for i in citados]
    for (ta, a), (tb, b) in combinations(itens, 2):
        if ta != tb and sobrepoe(a, b):
            return 'alto'
    if any(t == 'P' and x['probabilidade_maxima'] >= 0.5 for t, x in itens):
        return 'alto'
    if len(itens) == 1:
        t, x = itens[0]
        if t == 'P':
            return 'médio'
        if t == 'F' and x['sinais']:
            return 'médio'
        if t == 'C':
            return 'baixo'
    return None


def correspondem(a, b):
    """Duas indicações de chamadas diferentes correspondem: mesmo motivo e sobreposição de tempo."""
    return a.get('motivo') == b.get('motivo') and sobrepoe(a, b)


def estaveis(rodadas):
    """Para cada rodada, lista de booleanos: a indicação tem correspondente em todas as outras."""
    return [[all(any(correspondem(ind, outra) for outra in rodadas[s]) for s in range(len(rodadas)) if s != r)
             for ind in rodadas[r]] for r in range(len(rodadas))]


def alcancado(intervalo, indicacoes):
    """Algum intervalo das indicações tem tempo em comum com o intervalo dado."""
    return any(sobrepoe(intervalo, ind) for ind in indicacoes)


def mmss(s):
    """257.15 -> '4:17,2'"""
    return f'{int(s // 60)}:{s % 60:04.1f}'.replace('.', ',')

In [6]:
CHAMADAS = {(n, r): ler(os.path.join(PASTA_CHAMADAS, f'faixa{n:02d}_{r}.json')) for n in range(1, 6) for r in range(1, 4)}
RODADAS = {n: [(CHAMADAS[(n, r)]['resposta'] or {}).get('indicacoes', []) for r in range(1, 4)] for n in range(1, 6)}
ESTAVEIS = {n: estaveis(RODADAS[n]) for n in range(1, 6)}

linhas = []
for n in range(1, 6):
    fontes = mapa_fontes(ENTRADAS[n])
    for r in range(1, 4):
        for k, ind in enumerate(RODADAS[n][r - 1]):
            problemas = problemas_conformidade(ind)
            inexistentes, fora = invencao(ind, fontes)
            esperado = grau_esperado(ind, fontes)
            linhas.append({
                'faixa': n, 'rodada': r, 'n': k + 1, 'inicio': ind.get('inicio'), 'fim': ind.get('fim'),
                'fontes': ', '.join(ind.get('fontes') or []), 'motivo': ind.get('motivo'),
                'grau': ind.get('grau_de_apoio'), 'frase': ind.get('frase'),
                'conforme': not problemas, 'problemas': '; '.join(problemas),
                'ids_inexistentes': ', '.join(inexistentes), 'fora_do_intervalo': fora,
                'inventada': bool(inexistentes) or fora,
                'termos': ', '.join(termos_na_frase(ind.get('frase'))),
                'grau_esperado': esperado or '', 'grau_coerente': (ind.get('grau_de_apoio') == esperado) if esperado else None,
                'estavel': ESTAVEIS[n][r - 1][k],
            })
INDICACOES = pd.DataFrame(linhas)
INDICACOES.to_csv(os.path.join(PASTA_MEDIDAS, 'indicacoes.csv'), index=False)

print(pd.DataFrame([resumo(CHAMADAS[c]) for c in CHAMADAS]).to_string(index=False))
print('\n', len(INDICACOES), 'indicações nas quinze chamadas')

  chamada  indicações erro de leitura  tempo (s)  tokens entrada  tokens saida  tokens raciocinio  tokens total
faixa01_1          12                      51.32           10996          1673               5847         18516
faixa01_2          13                      51.26           10996          1548               3390         15934
faixa01_3          10                      61.21           10996          1451               6602         19049
faixa02_1           6                      39.69            7495           797               4720         13012
faixa02_2           7                      60.95            7495           903               7742         16140
faixa02_3           7                      38.45            7495           990               4033         12518
faixa03_1           4                     115.57           10216           411              15358         25985
faixa03_2           4                      50.81           10216           593               6210       

**Resultado.** As quinze respostas foram lidas como JSON, com 98 indicações ao todo. O tempo de raciocínio variou muito para a mesma entrada. Na faixa 04, a primeira chamada usou 35.698 tokens de raciocínio e levou 264 s, perto do limite de 5 minutos, e a terceira usou 2.936 tokens e levou 25 s.

## **7. Dossiês**

*O dossiê de cada faixa é montado pelo código a partir da primeira das três chamadas, em ordem de tempo, com os tempos, as fontes, o motivo, o grau e a frase tal como o modelo os devolveu (seção 5 do desenho). A marca “repete-se nas três chamadas” assinala as indicações que têm correspondente nas outras duas. Os resultados da conferência automática não entram no dossiê, para que a leitura da pesquisadora (seção 8 do desenho) se faça sobre o que o modelo produziu. A célula mostra o dossiê da faixa 04, e os cinco ficam em `resultados/fase5/dossies`.*

In [ ]:
def dossie_md(entrada, indicacoes, marcas, cabecalho):
    """Dossiê em Markdown, em ordem de tempo, montado a partir da saída do modelo."""
    linhas = [f'# Dossiê de escuta · faixa {entrada["faixa"]:02d}', '', cabecalho, '']
    ordem = sorted(range(len(indicacoes)), key=lambda k: (indicacoes[k]['inicio'], indicacoes[k]['fim']))
    if not ordem:
        linhas.append('O modelo não fez nenhuma indicação nesta chamada.')
    for n, k in enumerate(ordem, 1):
        ind = indicacoes[k]
        repete = ' · repete-se nas três chamadas' if marcas[k] else ''
        linhas += [f'## {n}. {mmss(ind["inicio"])} a {mmss(ind["fim"])}{repete}', '',
                   f'**Motivo** {ind["motivo"]} · **Grau de apoio** {ind["grau_de_apoio"]} · '
                   f'**Fontes** {", ".join(ind["fontes"])}', '', ind['frase'].strip(), '']
    return '\n'.join(linhas).rstrip() + '\n'


for n in range(1, 6):
    registro = CHAMADAS[(n, 1)]
    data = datetime.fromisoformat(registro['data_execucao']).strftime('%d/%m/%Y')
    cabecalho = (f'Duração da faixa {mmss(ENTRADAS[n]["duracao_s"])}. Montado pelo caderno 5b a partir da primeira '
                 f'de três chamadas ao modelo {registro["versao_modelo"] or registro["modelo"]}, feita em {data}. '
                 'As indicações seguem a ordem de tempo, e a marca “repete-se nas três chamadas” assinala as que '
                 'têm, nas outras duas chamadas, uma indicação com o mesmo motivo e algum tempo em comum.')
    texto = dossie_md(ENTRADAS[n], RODADAS[n][0], ESTAVEIS[n][0], cabecalho)
    with open(os.path.join(PASTA_DOSSIES, f'faixa{n:02d}.md'), 'w', encoding='utf-8') as f:
        f.write(texto)

print(sorted(os.listdir(PASTA_DOSSIES)), '\n')
print(open(os.path.join(PASTA_DOSSIES, 'faixa04.md'), encoding='utf-8').read())

['faixa01.md', 'faixa02.md', 'faixa03.md', 'faixa04.md', 'faixa05.md'] 

# Dossiê de escuta · faixa 04

Duração da faixa 1:04,9. Montado pelo caderno 5b a partir da primeira de três chamadas ao modelo gemini-3.8-flash, feita em 03/11/2026. As indicações seguem a ordem de tempo, e a marca “repete-se nas três chamadas” assinala as que têm, nas outras duas chamadas, uma indicação com o mesmo motivo e algum tempo em comum.

## 1. 0:03,0 a 0:05,0

**Motivo** rótulo isolado · **Grau de apoio** baixo · **Fontes** C-001

A pesquisadora pode verificar o trecho em que o modelo CLAP atribuiu a descrição 'a man giving a speech', que aparece uma única vez na faixa.

## 2. 0:07,7 a 0:09,0 · repete-se nas três chamadas

**Motivo** divergência entre transcrições · **Grau de apoio** médio · **Fontes** F-003

A pesquisadora pode verificar a fala no segmento em que o modelo registrou divergência entre transcrições para o texto 'Iniciando pela batucada.'.

## 3. 0:30,0 a 0:33,0 · repete-se nas três chamad

**Resultado.** Os cinco dossiês reúnem 32 indicações (12, 6, 4, 3 e 7, da faixa 01 à 05), com as frases tal como o modelo as devolveu, inclusive as aspas retas. A leitura e a escuta de conferência estão em [`02_leitura_e_escuta.md`](02_leitura_e_escuta.md).

## **8. Medidas**

*As oito medidas da seção 7 do desenho, cada uma gravada numa tabela em `resultados/fase5/medidas`.*

### 8.1 Conformidade

Proporção de indicações com início, fim, fonte, motivo, grau e frase válidos, por faixa, e chamadas cuja resposta não pôde ser lida como JSON.

In [7]:
def proporcao(serie):
    return f'{serie.mean():.0%}' if len(serie) else '·'


conf = INDICACOES.groupby('faixa').agg(indicacoes=('n', 'size'), conformes=('conforme', 'sum'))
conf['proporção'] = (conf['conformes'] / conf['indicacoes']).map('{:.0%}'.format)
conf.to_csv(os.path.join(PASTA_MEDIDAS, 'conformidade.csv'))
print(conf.to_string())
print('\nTotal:', int(INDICACOES['conforme'].sum()), 'de', len(INDICACOES), f"({proporcao(INDICACOES['conforme'])})")
print('Chamadas com erro de leitura:', [c for c in CHAMADAS if CHAMADAS[c]['erro_leitura']] or 'nenhuma')
print(INDICACOES.loc[~INDICACOES['conforme'], ['faixa', 'rodada', 'n', 'problemas']].to_string(index=False)
      if (~INDICACOES['conforme']).any() else 'Nenhuma indicação fora da conformidade.')

       indicacoes  conformes proporção
faixa                                 
1              35         35      100%
2              20         20      100%
3              12         12      100%
4              11         11      100%
5              20         20      100%

Total: 98 de 98 (100%)
Chamadas com erro de leitura: nenhuma
Nenhuma indicação fora da conformidade.


**Resultado.** As 98 indicações estão conformes. Como o esquema limitava motivos e graus às listas, a medida diz pouco sobre o modelo, e o que ela confirma é que nenhuma indicação veio sem tempo, fonte ou frase.

### 8.2 Invenção

Indicações que citam identificadores inexistentes ou cujo intervalo fica fora do intervalo das fontes citadas, com tolerância de 1 segundo.

In [8]:
inv = INDICACOES.groupby('faixa').agg(indicacoes=('n', 'size'),
                                     ids_inexistentes=('ids_inexistentes', lambda s: (s != '').sum()),
                                     fora_do_intervalo=('fora_do_intervalo', 'sum'), inventadas=('inventada', 'sum'))
inv.to_csv(os.path.join(PASTA_MEDIDAS, 'invencao.csv'))
print(inv.to_string())
print('\nTotal de indicações inventadas:', int(INDICACOES['inventada'].sum()), 'de', len(INDICACOES))
casos = INDICACOES[INDICACOES['inventada']]
if len(casos):
    print(casos[['faixa', 'rodada', 'n', 'inicio', 'fim', 'fontes', 'ids_inexistentes', 'fora_do_intervalo']].to_string(index=False))

       indicacoes  ids_inexistentes  fora_do_intervalo  inventadas
faixa                                                             
1              35                 0                  0           0
2              20                 0                  0           0
3              12                 0                  0           0
4              11                 0                  0           0
5              20                 0                  0           0

Total de indicações inventadas: 0 de 98


**Resultado.** Nenhuma indicação cita identificador inexistente ou tem intervalo fora das fontes citadas, contra a expectativa 2.

### 8.3 Restrição

Ocorrências, nas frases, de palavras iniciadas pelos termos da lista. Cada ocorrência é listada com a frase inteira, para a leitura da seção 8 do desenho.

In [9]:
ocorrencias = INDICACOES[INDICACOES['termos'] != ''][['faixa', 'rodada', 'n', 'termos', 'frase']]
ocorrencias.to_csv(os.path.join(PASTA_MEDIDAS, 'restricao.csv'), index=False)
print(len(ocorrencias), 'frases com termos da lista, de', len(INDICACOES))
for _, l in ocorrencias.iterrows():
    print(f"\nFaixa {l['faixa']:02d} · chamada {l['rodada']} · indicação {l['n']} · {l['termos']}\n   {l['frase']}")

0 frases com termos da lista, de 98


**Resultado.** Nenhuma frase usa os termos da lista. A leitura da pesquisadora, feita antes desta seção, também não marcou nenhuma frase como interpretação.

### 8.4 Coerência do grau de apoio

Proporção de indicações cujo grau corresponde à regra da seção 4, entre as que a regra permite avaliar, e tabela do grau atribuído pelo modelo em relação ao grau esperado.

In [10]:
avaliaveis = INDICACOES[INDICACOES['grau_esperado'] != '']
print(len(avaliaveis), 'indicações avaliáveis ·', len(INDICACOES) - len(avaliaveis), 'em combinações não previstas pela regra ou com identificador inexistente')
print('Coerência:', int(avaliaveis['grau_coerente'].sum()), 'de', len(avaliaveis), f"({proporcao(avaliaveis['grau_coerente'].astype(bool))})\n")
cruzada = pd.crosstab(INDICACOES['grau'], INDICACOES['grau_esperado'].replace('', 'não previsto'),
                      rownames=['atribuído'], colnames=['esperado'])
cruzada.to_csv(os.path.join(PASTA_MEDIDAS, 'grau.csv'))
print(cruzada.to_string())

97 indicações avaliáveis · 1 em combinações não previstas pela regra ou com identificador inexistente
Coerência: 97 de 97 (100%)

esperado   alto  baixo  médio  não previsto
atribuído                                  
alto         59      0      0             0
baixo         0     14      0             0
médio         0      0     24             1


**Resultado.** Todos os 97 graus avaliáveis seguem a regra. A indicação fora da regra reúne onze segmentos de fala da faixa 01 (3:29 a 3:59, chamada 2), combinação que a regra não prevê, com grau médio. A coerência não diz se o som está presente, e rótulos sem som correspondente na escuta dirigida da fase 4, como “Owl” (0,60), recebem grau alto, como manda a regra.

### 8.5 Estabilidade

Para cada faixa, o número de indicações em cada chamada e a proporção das indicações que têm correspondente nas outras duas chamadas (mesmo motivo e algum tempo em comum).

In [11]:
est = []
for n in range(1, 6):
    total = sum(len(r) for r in RODADAS[n])
    est.append({'faixa': n, **{f'chamada {r}': len(RODADAS[n][r - 1]) for r in range(1, 4)},
                'estáveis': sum(sum(e) for e in ESTAVEIS[n]),
                'proporção': f'{sum(sum(e) for e in ESTAVEIS[n]) / total:.0%}' if total else '·'})
est = pd.DataFrame(est).set_index('faixa')
est.to_csv(os.path.join(PASTA_MEDIDAS, 'estabilidade.csv'))
print(est.to_string())
print('\nMotivos por chamada:')
print(pd.crosstab([INDICACOES['faixa'], INDICACOES['rodada']], INDICACOES['motivo']).to_string())

       chamada 1  chamada 2  chamada 3  estáveis proporção
faixa                                                     
1             12         13         10        24       69%
2              6          7          7         9       45%
3              4          4          4        12      100%
4              3          4          4         6       55%
5              7          5          8        12       60%

Motivos por chamada:
motivo        coincidência entre fontes  divergência entre transcrições  evento sonoro apontado  repetição na transcrição  rótulo isolado  texto com indicação de ausência de fala  transcrição com baixa confiança
faixa rodada                                                                                                                                                                                                       
1     1                               4                               0                       4                         0               3    

**Resultado.** Das 98 indicações, 63 (64%) têm correspondente nas outras duas chamadas, de 45% na faixa 02 a 100% na faixa 03, cujas três chamadas produziram as mesmas quatro indicações. O número de indicações e a distribuição dos motivos variam pouco de uma chamada para outra, contra a expectativa 5. O motivo “repetição na transcrição” aparece uma única vez nas quinze chamadas, na faixa 01.

### 8.6 Cobertura

Proporção dos 44 eventos da tabela de referência da fase 4 e dos trechos das escutas dirigidas das fases 3 e 4 que têm algum tempo em comum com alguma indicação, na primeira chamada (a do dossiê) e em qualquer das três. Os trechos das escutas e suas categorias foram transcritos de `docs/fase3/02_escuta_dirigida.md` e `docs/fase4/03_escuta_dirigida.md`. Um trecho com mais de uma categoria conta em cada uma delas, e as categorias marcadas como “talvez” na fase 3 também contam. O trecho 29 da fase 4 (2:44 a 4:32 na faixa 05) é muito mais longo que os demais e reúne várias categorias.

In [12]:
def seg(t):
    """'1:25.6' ou '1:25,6' -> 85.6"""
    m, s = t.split(':')
    return int(m) * 60 + float(s.replace(',', '.'))


# Fase 3 · trecho: (faixa, início, fim, categorias) · categorias da fase 3 (1 a 7)
TRECHOS_FASE3 = {
    1: (1, '1:25.6', '1:29.8', {2}), 2: (1, '1:59.3', '2:01.2', {2}), 3: (1, '2:06.4', '2:14.8', {2}),
    4: (1, '2:27.6', '2:29.7', {2}), 5: (1, '2:55.3', '2:55.9', {2}), 6: (1, '2:58.8', '2:59.6', {2}),
    7: (1, '3:29.4', '3:29.6', {3}), 8: (1, '5:24.7', '5:35.3', {2}), 9: (2, '0:58.1', '0:58.7', {2, 3}),
    10: (2, '1:16.8', '1:17.0', {2}), 11: (2, '1:34.4', '1:35.7', {2}), 12: (2, '1:45.9', '2:06.4', {2}),
    13: (2, '2:09.0', '2:09.6', {2}), 14: (2, '2:14.2', '2:54.0', {2}), 15: (2, '3:17.0', '3:21.3', {2}),
    16: (3, '0:03.8', '0:04.8', {6}), 17: (3, '0:54.9', '0:55.1', {3}), 18: (3, '1:24.9', '1:26.2', {3}),
    19: (3, '2:23.3', '2:35.7', {3}), 20: (3, '2:40.7', '2:41.4', {2}), 21: (3, '2:54.7', '2:55.5', {2}),
    22: (3, '3:30.0', '3:30.3', {2}), 23: (3, '4:23.2', '4:24.5', {2}), 24: (3, '5:23.5', '5:23.9', {2}),
    25: (3, '5:59.4', '6:02.0', {5}), 26: (4, '0:43.6', '0:57.4', {2, 4}), 27: (4, '1:00.0', '1:04.2', {3}),
    28: (5, '0:00.0', '1:33.8', {4}), 29: (5, '1:58.8', '1:59.9', {2}), 30: (5, '3:49.1', '3:54.7', {2}),
    31: (5, '4:37.3', '4:46.7', {3, 4}),
}
CATEGORIAS_FASE3 = {1: 'fala real ausente da referência', 2: 'fala real deformada', 3: 'repetição',
                    4: 'texto sobre som não verbal', 5: 'texto sobre silêncio ou ruído', 6: 'efeito da marcação', 7: 'outro'}

# Fase 4 · trecho: (faixa, início, fim, categorias) · categorias da fase 4 (1 a 6)
TRECHOS_FASE4 = {
    1: (1, '0:00,0', '0:06,0', {1}), 2: (1, '1:34,0', '1:36,0', {3, 5}), 3: (1, '4:11,0', '4:13,0', {4}),
    4: (1, '4:30,0', '4:32,0', {3}), 5: (1, '4:54,0', '4:58,0', {3, 4}), 6: (1, '5:27,0', '5:35,5', {3, 5, 6}),
    7: (2, '0:02,0', '0:19,0', {5}), 8: (2, '0:29,0', '1:13,0', {3, 5}), 9: (2, '1:21,0', '1:37,0', {3, 5}),
    10: (2, '1:49,0', '1:59,0', {3}), 11: (2, '2:07,0', '2:35,0', {3, 5}), 12: (2, '2:43,0', '2:45,0', {3}),
    13: (2, '2:53,0', '3:17,0', {3, 5}), 14: (3, '0:10,0', '0:12,0', {5}), 15: (3, '2:12,0', '2:14,0', {3}),
    16: (3, '3:01,0', '3:03,0', {5}), 17: (3, '3:42,0', '3:44,0', {4}), 18: (3, '4:45,0', '4:48,0', {5}),
    19: (3, '5:38,0', '5:40,0', {5}), 20: (3, '5:57,0', '6:02,1', {1, 3, 5}), 21: (4, '0:00,0', '0:02,0', {3, 5}),
    22: (4, '0:11,0', '1:04,9', {2, 3, 4}), 23: (5, '0:18,0', '0:21,0', {4}), 24: (5, '1:09,0', '1:12,0', {2}),
    25: (5, '1:24,0', '1:26,0', {3}), 26: (5, '1:53,0', '2:01,0', {4}), 27: (5, '2:08,0', '2:13,0', {4}),
    28: (5, '2:25,0', '2:29,0', {4, 5}), 29: (5, '2:44,0', '4:32,0', {2, 3, 4, 5}), 30: (5, '4:44,0', '4:46,0', {4}),
    31: (5, '5:13,0', '5:15,0', {3}), 32: (5, '6:16,0', '6:18,0', {4}), 33: (5, '6:31,0', '6:33,0', {4}),
    34: (5, '7:15,0', '7:17,0', {4}),
}
CATEGORIAS_FASE4 = {1: 'som presente e não registrado', 2: 'som registrado, fora dos tempos', 3: 'som presente, com outro nome',
                    4: 'nenhum som correspondente', 5: 'efeito do filtro', 6: 'outro'}


def alcance(faixa, inicio, fim):
    """(alcançado na primeira chamada, número de chamadas que alcançam)"""
    intervalo = {'inicio': inicio, 'fim': fim}
    por_rodada = [alcancado(intervalo, RODADAS[faixa][r]) for r in range(3)]
    return por_rodada[0], sum(por_rodada)


with open(os.path.join(PASTA_REFERENCIAS, '01_eventos_referencia.md'), encoding='utf-8') as f:
    EVENTOS = ler_referencia(f.read())
ref = pd.DataFrame([{'evento': e['id'], 'faixa': e['faixa'], 'classe': e['classe'],
                     **dict(zip(['primeira', 'chamadas'], alcance(e['faixa'], e['inicio'], e['fim'])))} for e in EVENTOS])
ref.to_csv(os.path.join(PASTA_MEDIDAS, 'cobertura_referencia.csv'), index=False)
print(len(ref), 'eventos de referência · alcançados na primeira chamada:', int(ref['primeira'].sum()),
      f"({proporcao(ref['primeira'])}) · em alguma das três:", int((ref['chamadas'] > 0).sum()), f"({proporcao(ref['chamadas'] > 0)})")
print(ref.groupby('faixa').agg(eventos=('evento', 'size'), primeira=('primeira', 'sum'),
                               alguma=('chamadas', lambda s: (s > 0).sum())).to_string(), '\n')

esc = []
for fase, trechos, nomes in [(3, TRECHOS_FASE3, CATEGORIAS_FASE3), (4, TRECHOS_FASE4, CATEGORIAS_FASE4)]:
    for t, (faixa, ini, fim, cats) in trechos.items():
        primeira, chamadas = alcance(faixa, seg(ini), seg(fim))
        for c in sorted(cats):
            esc.append({'fase': fase, 'trecho': t, 'faixa': faixa, 'categoria': f'{c}. {nomes[c]}',
                        'primeira': primeira, 'chamadas': chamadas})
esc = pd.DataFrame(esc)
esc.to_csv(os.path.join(PASTA_MEDIDAS, 'cobertura_escutas.csv'), index=False)
tab = esc.groupby(['fase', 'categoria']).agg(trechos=('trecho', 'size'), primeira=('primeira', 'sum'),
                                              alguma=('chamadas', lambda s: (s > 0).sum()))
print(tab.to_string())

44 eventos de referência · alcançados na primeira chamada: 26 (59%) · em alguma das três: 27 (61%)
       eventos  primeira  alguma
faixa                           
1           13         9       9
2            5         5       5
3            5         2       2
4            7         2       2
5           14         8       9 

                                         trechos  primeira  alguma
fase categoria                                                    
3    2. fala real deformada                   22         6       7
     3. repetição                              7         1       1
     4. texto sobre som não verbal             3         1       1
     5. texto sobre silêncio ou ruído          1         1       1
     6. efeito da marcação                     1         0       0
4    1. som presente e não registrado          2         2       2
     2. som registrado, fora dos tempos        3         2       2
     3. som presente, com outro nome          17        10      1

**Resultado.** Três dos 44 eventos (1.13, 3.5 e 5.14) ocupam a faixa inteira e são alcançados por qualquer indicação. Sem eles, a primeira chamada alcança 23 de 41 eventos (56%). Ficam de fora as pausas e dois aplausos da faixa 01, as viradas de página da faixa 03, o surdo, o tamborim, os anúncios e os pulos da faixa 04 e, na faixa 05, o vento, a música de suspense e o salto de intensidade. Nas escutas dirigidas, a primeira chamada alcança 9 dos 31 trechos da fase 3 e 17 dos 34 da fase 4, e na fase 4 a proporção é parecida entre os trechos em que o rótulo nomeava um som presente (categoria 3, 10 de 17) e os trechos sem som correspondente (categoria 4, 7 de 13).

### 8.7 Faixa 05 e a escuta cega

Para cada uma das doze passagens da escuta cega da fase 1 (`docs/fase1/02_escuta_cega_faixa05.md`), em quantas das três chamadas alguma indicação tem tempo em comum com a passagem, e com que motivos e fontes isso ocorre na primeira chamada. Como a escuta cega marca segundos inteiros, cada passagem vai do início do primeiro segundo ao fim do último.

In [13]:
ESCUTA_CEGA = [
    ('00:00', '00:03', 'ruído da superfície do disco'), ('00:04', '01:03', 'música (religiosa?)'),
    ('01:04', '01:26', 'vento'), ('01:27', '01:51', 'música (sertaneja?) e vento'),
    ('01:52', '02:49', 'voz narrando, música e vento'), ('02:50', '03:14', 'vozes femininas cantando'),
    ('03:19', '03:50', 'vento'), ('03:52', '04:08', 'som metálico ressonante, com batidas'),
    ('04:20', '04:27', 'passos'), ('04:28', '04:37', 'vento e ruído do disco'),
    ('04:38', '05:03', 'silêncio de falas, música de suspense'), ('05:04', '07:21', 'voz gritando e música'),
]
cega = []
for ini, fim, oque in ESCUTA_CEGA:
    intervalo = {'inicio': seg(ini), 'fim': seg(fim) + 1}
    _, chamadas = alcance(5, intervalo['inicio'], intervalo['fim'])
    primeira = [i for i in RODADAS[5][0] if sobrepoe(intervalo, i)]
    cega.append({'passagem': f'{ini}-{fim}', 'o que soa (escuta cega)': oque, 'chamadas': chamadas,
                 'motivos (1ª chamada)': '; '.join(sorted({i['motivo'] for i in primeira})),
                 'fontes (1ª chamada)': ', '.join(sorted({f[0] for i in primeira for f in i['fontes']}))})
cega = pd.DataFrame(cega)
cega.to_csv(os.path.join(PASTA_MEDIDAS, 'faixa05_escuta_cega.csv'), index=False)
print(cega.to_string(index=False))

   passagem               o que soa (escuta cega)  chamadas      motivos (1ª chamada) fontes (1ª chamada)
00:00-00:03          ruído da superfície do disco         1                                              
00:04-01:03                   música (religiosa?)         3 coincidência entre fontes             C, F, P
01:04-01:26                                 vento         0                                              
01:27-01:51           música (sertaneja?) e vento         0                                              
01:52-02:49          voz narrando, música e vento         3 coincidência entre fontes                F, P
02:50-03:14              vozes femininas cantando         2    evento sonoro apontado                   P
03:19-03:50                                 vento         0                                              
03:52-04:08  som metálico ressonante, com batidas         3    evento sonoro apontado                   P
04:20-04:27                                pas

**Resultado.** A medida conta como alcançada a passagem que tem algum tempo em comum com uma indicação, mesmo que a indicação trate de outra coisa. Conferido o conteúdo, a música da abertura, a voz com música (1:52 a 2:49) e os gritos com música (5:04 a 7:21) são indicados, e o ruído da abertura aparece em uma das três chamadas. O vento não é indicado em nenhuma das três passagens em que soa sozinho ou com música. As vozes femininas (2:50 a 3:14) só são alcançadas por uma indicação de “Owl” e “Hoot”, o som metálico (3:52 a 4:08), por uma indicação de “Cattle, bovinae” e “Moo”, e os passos (4:20 a 4:27) e o vento com ruído (4:28 a 4:37), por uma indicação de “ocean waves” (4:26 a 4:28).

### 8.8 Tempo e custo

Tempo de resposta e tokens de cada chamada, com o custo equivalente na camada paga. Os preços abaixo (dólares por milhão de tokens, com os tokens de raciocínio cobrados como saída) foram consultados em 02/11/2026 numa fonte secundária e devem ser conferidos na página de preços do Google na data da execução.

In [14]:
PRECO_ENTRADA = 0.75  # US$ por milhão de tokens de entrada, até 31/12/2026
PRECO_SAIDA = 3.75    # US$ por milhão de tokens de saída, incluídos os de raciocínio

tc = []
for (n, r), reg in CHAMADAS.items():
    t = reg['tokens']
    saida = (t['saida'] or 0) + (t['raciocinio'] or 0)
    tc.append({'faixa': n, 'rodada': r, 'tempo (s)': reg['tempo_resposta_s'], 'entrada': t['entrada'],
               'saída': t['saida'], 'raciocínio': t['raciocinio'],
               'custo (US$)': round((t['entrada'] or 0) * PRECO_ENTRADA / 1e6 + saida * PRECO_SAIDA / 1e6, 4)})
tc = pd.DataFrame(tc)
tc.to_csv(os.path.join(PASTA_MEDIDAS, 'tempo_custo.csv'), index=False)
print(tc.to_string(index=False))
print(f"\nTempo médio {tc['tempo (s)'].mean():.1f} s · tokens de entrada {tc['entrada'].sum()} · "
      f"de saída {tc['saída'].sum()} · de raciocínio {tc['raciocínio'].fillna(0).sum():.0f} · "
      f"custo equivalente das quinze chamadas US$ {tc['custo (US$)'].sum():.2f}")

 faixa  rodada  tempo (s)  entrada  saída  raciocínio  custo (US$)
     1       1      51.32    10996   1673        5847       0.0364
     1       2      51.26    10996   1548        3390       0.0268
     1       3      61.21    10996   1451        6602       0.0384
     2       1      39.69     7495    797        4720       0.0263
     2       2      60.95     7495    903        7742       0.0380
     2       3      38.45     7495    990        4033       0.0245
     3       1     115.57    10216    411       15358       0.0668
     3       2      50.81    10216    593        6210       0.0332
     3       3      63.53    10216    577        8163       0.0404
     4       1     263.61     2519    369       35698       0.1371
     4       2      47.46     2519    490        6000       0.0262
     4       3      24.79     2519    502        2936       0.0148
     5       1      31.62    16185    989        3229       0.0280
     5       2      30.13    16185    704        3190       0.

**Resultado.** As quinze chamadas custaram o equivalente a US$ 0,60, a maior parte em tokens de raciocínio (118.189, contra 13.203 de saída). O tempo médio de resposta foi de 65 s.

## **9. Ambiente de execução**

In [ ]:
import importlib.metadata as md
print(datetime.now().strftime('%d/%m/%Y %H:%M'))
print('Python', sys.version.split()[0])
for p in ['pandas', 'google-genai']:
    print(p, md.version(p))

03/11/2026 17:32
Python 3.13.18
pandas 2.2.3
google-genai 2.12.1


## **10. Testes automáticos**

*Executa todos os testes de `tests/`, os das fases 3 e 4, os de `tests/test_entradas.py` e os de `tests/test_dossie.py`, que conferem, com casos de resposta conhecida, a identidade entre a instrução do código e o anexo A do desenho, o esquema, a conformidade, a invenção (identificadores e tempos), os termos interpretativos, o grau esperado, a estabilidade, a cobertura e a ordem do dossiê. As funções testadas estão em `src/dossie.py`, cópia idêntica das definidas neste caderno.*

In [ ]:
!pip install -q jiwer==4.0.0
!cd /content/drive/MyDrive/escuta-cerrada-poc && python -m pytest -q tests

.....................................                                    [100%]
37 passed in 2.61s


**Resultado.** Os 37 testes passaram (03/11/2026).